In [ ]:
import pandas as pd
import os
import torch
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

#

# Model - 1 : Chart Summary
Qwen2.5-VL-7B-Instruct

In [ ]:
VL_MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"


qwen_processor = AutoProcessor.from_pretrained(VL_MODEL_NAME)


if 'qwen_model' not in globals() or qwen_model is None:
    qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        VL_MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    qwen_model.eval()
    print("Qwen Model loaded.")
else:
    print("Qwen Model already loaded, skipping.")

# Model - 2: Translation Model NLLB200

facebook/nllb-200-distilled-600M

In [ ]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

TRANS_MODEL_NAME = "facebook/nllb-200-distilled-600M"

trans_tokenizer = AutoTokenizer.from_pretrained(TRANS_MODEL_NAME)
if 'trans_model' not in globals() or trans_model is None:
  trans_model = AutoModelForSeq2SeqLM.from_pretrained(TRANS_MODEL_NAME)
  trans_model.eval()
  print("NLLB translation model loaded.")
else:
  print("NLLB translation model already loaded, skipping.")

def translate_en_to_bn(text):
    inputs = trans_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

    outputs = trans_model.generate(
        **inputs,
        forced_bos_token_id=trans_tokenizer.convert_tokens_to_ids("ben_Beng"),
        max_new_tokens=256
    )

    return trans_tokenizer.decode(outputs[0], skip_special_tokens=True)


def summarize_and_translate(summary_en):
    return translate_en_to_bn(summary_en)

## Pipeline: (Planner+insight_extractor+summarizer) = Qwen2.5-VL-7B-Instruct

Part - 2: Using Single Prompt For *Panner, Extactor, Summarizer* **AND** Single Call

In [ ]:
def generate_response_optimized(image, prompt_text, max_new_tokens=300):
    """Encode image ONCE, reuse for generation."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]
    text = qwen_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = qwen_processor(
        text=[text],
        images=[image],
        return_tensors="pt"
    ).to(device)

    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy = faster than sampling
            use_cache=True,           # KV cache reuse
            temperature=None,         # disable when do_sample=False
            top_p=None,               # disable when do_sample=False
        )

    new_tokens = output[0][input_len:]
    return qwen_processor.decode(new_tokens, skip_special_tokens=True).strip()


# ── Chain-of-Thought Single-Pass Prompt ──────────────────────────────────────
SINGLE_PASS_PROMPT = """You are an expert data analyst and data journalist.

Analyze the chart image carefully.

Think silently through these steps (DO NOT output them):
1. Identify chart type, domain, axes, units, and time range (if present)
2. Extract key data insights (trends, comparisons, patterns, anomalies)
3. Interpret domain meaning (causes, implications, real-world impact)

Then produce ONLY the final answer:

Summary:
Write a single, well-structured paragraph (4–6 sentences) that:
- Starts with the main trend or takeaway
- Includes at least one comparison or pattern
- Mentions any anomaly or notable feature (if present)
- Explains the real-world significance

Use clear, confident, natural English. No bullet points. No extra text."""


def run_pipeline_fast(image):
    print("Running single-pass optimized pipeline...")
    summary = generate_response_optimized(image, SINGLE_PASS_PROMPT, max_new_tokens=250)
    print("\n── Chart Summary ──")
    print(summary)
    return {"summary": summary}

# Load Old Dataset

In [ ]:
folder_path = "/content/drive/MyDrive/TWP/Image_Dataset/TestSet/UthfolToDO-2"

In [ ]:
image_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
dataset = [Image.open(f).convert("RGB") for f in image_files]

print(f"Loaded {len(dataset)} images into the dataset.")



In [ ]:
df_images = pd.DataFrame({"image": image_files})
df_images['image_name'] = df_images['image'].apply(lambda x: os.path.basename(x))
# Drop the 'image' column (full path) as it's not needed, keeping only 'image_name'
df_images = df_images.drop(columns=['image'])
print(f"Size: {len(df_images)}")


In [ ]:
df_images.head()

## New Dataset Creation by Single Prompt technique

In [ ]:

summary_data = []

for i, image in enumerate(dataset):
    print(f"\nProcessing image {i+1}/{len(dataset)}...")

    # Generate English summary
    english_summary_result = run_pipeline_fast(image)
    english_summary = english_summary_result["summary"]

    # Generate Bangla summary
    bangla_summary = summarize_and_translate(english_summary)

    summary_data.append({
        "english_summary": english_summary,
        "bangla_summary": bangla_summary
    })

# Create a Pandas DataFrame
df_by_single_prompt = pd.DataFrame(summary_data)
df_by_single_prompt.head()


# Create Final Dataset & Save

In [ ]:
final_dataset = pd.concat([df_images, df_by_single_prompt], axis=1)
print("Final dataset created:")
final_dataset.head()


In [ ]:
output_folder_path = "/content/drive/MyDrive/TWP/Summary_Dataset/TestSet"

# Ensure the directory exists
os.makedirs(output_folder_path, exist_ok=True)

# Count existing .xlsx files in the directory
existing_files = [f for f in os.listdir(output_folder_path) if f.endswith('.xlsx')]
next_file_number = len(existing_files) + 1

# Create the new unique filename
output_file_name = f"testset_summaries_{next_file_number}.xlsx"
output_file_path = os.path.join(output_folder_path, output_file_name)

final_dataset.to_excel(output_file_path, index=False)
print(f"\nFinal dataset saved to {output_file_path}")